In [1]:
"""
03_opentelemetry_xray.py — OpenTelemetry spans and state x-ray.

Run from repository root:
    uv run python packages/aoa-action-machine/examples/03_opentelemetry_xray.py

Requires:
    pip install "aoa-action-machine[otel]"
"""

import asyncio

from opentelemetry.sdk._logs import LoggerProvider
from opentelemetry.sdk._logs.export import InMemoryLogRecordExporter, SimpleLogRecordProcessor
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter
from pydantic import Field

from aoa.action_machine.auth import NoneRole
from aoa.action_machine.context import Context
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.aspects import regular_aspect, summary_aspect
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.checkers import result_float, result_int, result_string
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.model import BaseAction, BaseParams, BaseResult
from aoa.action_machine.plugin.open_telemetry import OpenTelemetryPlugin
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine


class OrderDomain(BaseDomain):
    name = "orders"
    description = "Order processing domain"


class CreateDraftParams(BaseParams):
    raw_sku: str = Field(description="SKU exactly as received from a client")
    quantity: int = Field(description="Requested quantity")
    unit_price: float = Field(description="Unit price for this request")


class CreateDraftResult(BaseResult):
    sku: str = Field(description="Normalised SKU")
    quantity: int = Field(description="Validated quantity")
    total: float = Field(description="Line total")
    message: str = Field(description="Human-readable summary")


@meta(description="Create order draft with telemetry", domain=OrderDomain)
@check_roles(NoneRole)
class CreateDraftTelemetryAction(BaseAction[CreateDraftParams, CreateDraftResult]):

    @regular_aspect("Normalise SKU")
    @result_string("sku", required=True, min_length=3)
    async def normalise_aspect(self, params, state, box, connections):
        return {"sku": params.raw_sku.strip().upper()}

    @regular_aspect("Calculate total")
    @result_string("sku", required=True, min_length=3)
    @result_int("quantity", required=True, min_value=1)
    @result_float("total", required=True, min_value=0.01)
    async def calculate_aspect(self, params, state, box, connections):
        quantity = params.quantity
        total = round(quantity * params.unit_price, 2)
        return {
            "sku": state["sku"],
            "quantity": quantity,
            "total": total,
        }

    @summary_aspect("Build draft result")
    async def build_summary(self, params, state, box, connections):
        return CreateDraftResult(
            sku=state["sku"],
            quantity=state["quantity"],
            total=state["total"],
            message=f"{state['quantity']}x {state['sku']} = ${state['total']}",
        )


async def main() -> None:
    span_exporter = InMemorySpanExporter()
    tracer_provider = TracerProvider()
    tracer_provider.add_span_processor(SimpleSpanProcessor(span_exporter))

    log_exporter = InMemoryLogRecordExporter()
    logger_provider = LoggerProvider()
    logger_provider.add_log_record_processor(SimpleLogRecordProcessor(log_exporter))

    plugin = OpenTelemetryPlugin(
        tracer_provider=tracer_provider,
        logger_provider=logger_provider,
        service_name="orders-service",
    )
    machine = ActionProductMachine(plugins=[plugin])

    result = await machine.run(
        Context(),
        CreateDraftTelemetryAction(),
        CreateDraftParams(raw_sku="  sku-42  ", quantity=3, unit_price=19.99),
    )
    print(f"Result: {result.message}")

    print("Spans:")
    for span in span_exporter.get_finished_spans():
        print(f"  {span.name}")

    print("State x-ray logs:")
    for record in log_exporter.get_finished_logs():
        attrs = dict(record.log_record.attributes or {})
        state_attrs = {
            key: value
            for key, value in attrs.items()
            if str(key).startswith("aoa.state.")
        }
        if state_attrs:
            aspect = attrs.get("aoa.aspect")
            print(f"  {aspect}: {state_attrs}")


asyncio.run(main())


Result: 3x SKU-42 = $59.97
Spans:
  normalise_aspect
  calculate_aspect
  build_summary
  CreateDraftTelemetryAction
State x-ray logs:
  normalise_aspect: {'aoa.state.sku': 'SKU-42'}
  calculate_aspect: {'aoa.state.sku': 'SKU-42', 'aoa.state.quantity': 3, 'aoa.state.total': 59.97}
